# Peran Plotly dalam Machine Learning
Notebook ini menjelaskan bagaimana Plotly dapat membantu setiap tahap Machine Learning: eksplorasi data, analisis fitur, evaluasi model, dan interpretasi hasil kasus : prediksi diagnosa penyakit jantung. 

## 1. Persiapan Data dan Eksplorasi
Kita buat dataset sintetis untuk kasus prediksi penyakit jantung, lalu gunakan Plotly untuk memahami pola data sebelum melatih model.

### Peran Pandas dan Numpy dalam Data collection 

In [1]:
import numpy as np
import pandas as pd


np.random.seed(42)
num_samples = 1000
data_ml = {
    'usia': np.random.randint(30, 75, num_samples),
    'jenis_kelamin': np.random.choice([0, 1], num_samples, p=[0.4, 0.6]),
    'kolesterol': np.random.randint(150, 300, num_samples),
    'tekanan_darah': np.random.randint(100, 180, num_samples),
    'detak_jantung_maks': np.random.randint(120, 200, num_samples),
    'angina_saat_olahraga': np.random.choice([0, 1], num_samples, p=[0.7, 0.3])
}
df = pd.DataFrame(data_ml)
heart_disease_risk = (
    0.3 * (df['usia'] / df['usia'].max()) +
    0.2 * df['jenis_kelamin'] +
    0.25 * (df['kolesterol'] / df['kolesterol'].max()) +
    0.2 * (df['tekanan_darah'] / df['tekanan_darah'].max()) +
    0.1 * df['angina_saat_olahraga'] -
    0.15 * (df['detak_jantung_maks'] / df['detak_jantung_maks'].max())
)
df['penyakit_jantung'] = (heart_disease_risk > np.percentile(heart_disease_risk, 60)).astype(int)
display(df.head())

,usia,jenis_kelamin,kolesterol,tekanan_darah,detak_jantung_maks,angina_saat_olahraga,penyakit_jantung
0,68,1,273,151,167,1,1
1,58,1,212,119,125,0,1
2,44,0,265,131,175,1,0
3,72,1,211,113,178,0,1
4,37,0,156,149,148,1,0


### Peran Plotly dalam Visualisasi Data

In [2]:
import plotly.express as px
import plotly.graph_objects as go


fig_age_chol = px.scatter(
    df, x='usia', y='kolesterol', color='penyakit_jantung',
    color_continuous_scale=['#1f77b4', '#ff7f0e'],
    labels={'usia': 'Usia', 'kolesterol': 'Kolesterol', 'penyakit_jantung': 'Penyakit Jantung'},
    title='Sebaran Usia vs Kolesterol Berdasarkan Status Penyakit Jantung'
)
fig_age_chol.update_traces(marker=dict(size=8, line=dict(width=0.5, color='DarkSlateGrey')))
fig_age_chol.show()

fig_box = px.box(
    df, x='penyakit_jantung', y='tekanan_darah',
    labels={'penyakit_jantung': 'Penyakit Jantung', 'tekanan_darah': 'Tekanan Darah'},
    title='Distribusi Tekanan Darah pada Kelompok Pasien'
)
fig_box.update_layout(xaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=['Tidak', 'Ya']))
fig_box.show()

## 2. Pelatihan Model dan Evaluasi
Kita latih model Regresi Logistik sederhana, lalu gunakan Plotly untuk memperlihatkan matriks kebingungan dan kurva ROC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc

X = df.drop('penyakit_jantung', axis=1)
y = df['penyakit_jantung']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
print(f'Akurasi Model: {accuracy:.2f}')

fig_cm = px.imshow(
    cm,
    x=['Prediksi Tidak', 'Prediksi Ya'],
    y=['Aktual Tidak', 'Aktual Ya'],
    color_continuous_scale='Blues',
    text_auto=True,
    labels={'x': 'Prediksi', 'y': 'Aktual'},
    title='Matriks Kebingungan Model Regresi Logistik'
)
fig_cm.update_layout(xaxis_title='Prediksi Model', yaxis_title='Label Aktual')
fig_cm.show()

y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC curve (AUC = {roc_auc:.2f})', line=dict(color='#1f77b4')))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(color='gray', dash='dash'), showlegend=False))
fig_roc.update_layout(
    title='Kurva ROC untuk Evaluasi Model',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    template='plotly_white'
)
fig_roc.show()

Akurasi Model: 0.90


## 3. Interpretasi Model dengan Visualisasi Fitur
Plotly dapat membantu menjelaskan pengaruh fitur terhadap model dengan visualisasi koefisien dan kontribusi.

In [ ]:
feature_importance = pd.DataFrame({
    'Fitur': X.columns,
    'Koefisien': model.coef_[0]
}).sort_values(by='Koefisien', ascending=False)

fig_imp = px.bar(
    feature_importance, x='Fitur', y='Koefisien', color='Koefisien',
    color_continuous_scale=px.colors.sequential.RdBu,
    title='Koefisien Fitur Model Regresi Logistik',
    labels={'Koefisien': 'Pengaruh pada Log Odds'}
)
fig_imp.update_layout(showlegend=False, template='plotly_white')
fig_imp.show()

new_patient = pd.DataFrame([
    {
        'usia': 65,
        'jenis_kelamin': 1,
        'kolesterol': 280,
        'tekanan_darah': 160,
        'detak_jantung_maks': 140,
        'angina_saat_olahraga': 1
    }
])

contributions = new_patient.iloc[0] * model.coef_[0]
contrib_df = pd.DataFrame({
    'Fitur': X.columns,
    'Kontribusi': contributions.values
}).sort_values(by='Kontribusi')

fig_contrib = px.bar(
    contrib_df, x='Kontribusi', y='Fitur', orientation='h',
    title='Kontribusi Fitur pada Prediksi Pasien Baru',
    labels={'Kontribusi': 'Kontribusi pada Log Odds', 'Fitur': 'Fitur'},
    color='Kontribusi',
    color_continuous_scale=px.colors.sequential.RdBu
)
fig_contrib.update_layout(showlegend=False, template='plotly_white')
fig_contrib.show()

## Kesimpulan
Plotly membantu dalam Machine Learning dengan membuat data eksploratori menjadi interaktif, memperlihatkan pola fitur, dan menjelaskan hasil model secara visual. Notebook ini menampilkan contoh nyata untuk memahami bagaimana visualisasi bisa meningkatkan pemahaman sebelum dan sesudah pelatihan model.